In [ ]:
import pandas as pd
import geopandas as gpd
from shapely import wkt

In [ ]:
pools = gpd.read_file('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/public_swimmingpools_germany/clean_datasets/pois_within_shapefile.gpkg')

pools.info()

In [ ]:
groundtruth_pools  = pd.read_csv("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/groundtruth_pools/baederleben_groundtruth_pools.csv")
groundtruth_pools['address'] = groundtruth_pools['street'] + ', ' + groundtruth_pools['postal_code'].astype(str) + ' ' + groundtruth_pools['city'] + ', Germany'
groundtruth_pools.info()

### Mapping google pois to ground truth obtained from Baederleben.de

In [ ]:
import pandas as pd
import geopandas as gpd
import re
from difflib import SequenceMatcher

def conservative_german_address_clean(address: str) -> str:
    """Clean addresses conservatively for exact matching"""
    if pd.isna(address):
        return ""
    
    address = str(address).strip()
    address = re.sub(r',\s*germany\s*$', '', address, flags=re.IGNORECASE)
    
    umlaut_replacements = {'ä': 'ae', 'ö': 'oe', 'ü': 'ue', 'ß': 'ss'}
    for old, new in umlaut_replacements.items():
        address = address.replace(old, new)
    
    street_standardizations = {
        r'\bstr\.\b': 'strasse',
        r'\bstr\b': 'strasse',
        r'\bstraße\b': 'strasse',
    }
    for pattern, replacement in street_standardizations.items():
        address = re.sub(pattern, replacement, address, flags=re.IGNORECASE)
    
    address = re.sub(r'[^\w\s]', ' ', address)
    address = re.sub(r'\s+', ' ', address).strip()
    
    return address.lower()

def extract_postal_code(address: str) -> str:
    """Extract postal code for faster grouping"""
    match = re.search(r'(\d{5})', address)
    return match.group(1) if match else ''

def extract_street_core(address: str) -> str:
    """
    Extract core street name by removing house numbers and postal codes
    """
    # Remove postal code
    address_no_postal = re.sub(r'\d{5}', '', address)
    
    # Remove standalone numbers (likely house numbers)
    address_no_numbers = re.sub(r'\b\d+[a-z]?\b', '', address_no_postal)
    
    # Remove extra whitespace and return
    return re.sub(r'\s+', ' ', address_no_numbers).strip()

def strict_close_match(addr1: str, addr2: str, postal1: str, postal2: str) -> tuple[bool, float, str]:
    """
    STRICT Option A criteria for close matches:
    - Same postal code required
    - Similarity > 0.85
    - Must share core street name
    """
    # Must have same postal code
    if not postal1 or not postal2 or postal1 != postal2:
        return False, 0.0, "different_postal_codes"
    
    # Calculate overall similarity
    overall_similarity = SequenceMatcher(None, addr1, addr2).ratio()
    
    # Must have high overall similarity
    if overall_similarity <= 0.85:
        return False, overall_similarity, f"similarity_too_low_{overall_similarity:.2f}"
    
    # Extract core street names (without numbers and postal codes)
    street_core1 = extract_street_core(addr1)
    street_core2 = extract_street_core(addr2)
    
    # Calculate street core similarity
    street_similarity = SequenceMatcher(None, street_core1, street_core2).ratio()
    
    # Must share core street name (high similarity in street part)
    if street_similarity < 0.8:
        return False, overall_similarity, f"street_core_different_{street_similarity:.2f}"
    
    # All criteria passed - this is a close match
    reason = f"strict_match_similarity_{overall_similarity:.2f}_street_{street_similarity:.2f}"
    return True, overall_similarity, reason

# Load datasets
print("Loading datasets...")
pools = gpd.read_file('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/public_swimmingpools_germany/clean_datasets/pois_within_shapefile.gpkg')
groundtruth_pools = pd.read_csv("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/groundtruth_pools/baederleben_groundtruth_pools.csv")
groundtruth_pools['address'] = groundtruth_pools['street'] + ', ' + groundtruth_pools['postal_code'].astype(str) + ' ' + groundtruth_pools['city'] + ', Germany'

print(f"Original Google Maps pools: {len(pools)}")
print(f"Original Ground Truth pools: {len(groundtruth_pools)}")

# Clean addresses
print("\nCleaning addresses...")
pools_clean = pools.copy()
groundtruth_clean = groundtruth_pools.copy()

pools_clean['cleaned_address'] = pools_clean['address'].apply(conservative_german_address_clean)
groundtruth_clean['cleaned_address'] = groundtruth_clean['address'].apply(conservative_german_address_clean)

# Add postal codes for faster grouping
pools_clean['postal_code'] = pools_clean['cleaned_address'].apply(extract_postal_code)
groundtruth_clean['postal_code'] = groundtruth_clean['cleaned_address'].apply(extract_postal_code)

print("Postal code extraction complete")

# Step 1: Exact matches (fast)
print("Finding exact matches...")
exact_matches_mask_google = pools_clean['cleaned_address'].isin(groundtruth_clean['cleaned_address'])
exact_matches_mask_groundtruth = groundtruth_clean['cleaned_address'].isin(pools_clean['cleaned_address'])

exact_matches_google = pools_clean[exact_matches_mask_google].copy()
exact_matches_groundtruth = groundtruth_clean[exact_matches_mask_groundtruth].copy()

exact_matches = pd.merge(
    exact_matches_google,
    exact_matches_groundtruth,
    on='cleaned_address',
    suffixes=('_google', '_groundtruth')
)
exact_matches['match_type'] = 'exact'
exact_matches['similarity_score'] = 1.0
exact_matches['match_reason'] = 'exact_match'

print(f"Exact matches found: {len(exact_matches)}")

# Step 2: Get remaining addresses for close matching
remaining_google = pools_clean[~exact_matches_mask_google].copy()
remaining_groundtruth = groundtruth_clean[~exact_matches_mask_groundtruth].copy()

print(f"Remaining for close matching: {len(remaining_google)} Google, {len(remaining_groundtruth)} Ground Truth")

# Step 3: STRICT close matches
print("Finding STRICT close matches (same postal code + similarity > 0.85 + same street core)...")
close_matches = []

# Group by postal code to reduce comparisons
google_by_postal = remaining_google.groupby('postal_code')
groundtruth_by_postal = {code: group for code, group in remaining_groundtruth.groupby('postal_code')}

processed = 0
total_groups = len(google_by_postal)

for postal_code, google_group in google_by_postal:
    processed += 1
    if processed % 100 == 0:
        print(f"Processed {processed}/{total_groups} postal code groups...")
    
    if postal_code in groundtruth_by_postal and postal_code:  # Only if postal code exists
        groundtruth_group = groundtruth_by_postal[postal_code]
        
        # Compare addresses within same postal code only
        for _, google_row in google_group.iterrows():
            google_addr = google_row['cleaned_address']
            google_postal = google_row['postal_code']
            
            best_match = None
            best_similarity = 0
            
            for _, groundtruth_row in groundtruth_group.iterrows():
                groundtruth_addr = groundtruth_row['cleaned_address']
                groundtruth_postal = groundtruth_row['postal_code']
                
                # Apply STRICT criteria
                is_close, similarity, reason = strict_close_match(
                    google_addr, groundtruth_addr, google_postal, groundtruth_postal
                )
                
                if is_close and similarity > best_similarity:
                    best_similarity = similarity
                    best_match = (groundtruth_row, reason)
            
            if best_match is not None:
                groundtruth_row, reason = best_match
                close_match_row = {}
                
                # Add Google data
                for col in google_row.index:
                    close_match_row[f'{col}_google'] = google_row[col]
                
                # Add Ground Truth data  
                for col in groundtruth_row.index:
                    close_match_row[f'{col}_groundtruth'] = groundtruth_row[col]
                
                close_match_row['similarity_score'] = best_similarity
                close_match_row['match_type'] = 'close_strict'
                close_match_row['match_reason'] = reason
                close_match_row['data_source'] = 'both'
                
                close_matches.append(close_match_row)

# Convert close matches to DataFrame
if close_matches:
    close_matches_df = pd.DataFrame(close_matches)
    print(f"STRICT close matches found: {len(close_matches_df)}")
else:
    close_matches_df = pd.DataFrame()
    print("No STRICT close matches found")

# Step 4: Create "only" datasets
print("Creating 'only' datasets...")

google_only = remaining_google.copy()
if not close_matches_df.empty:
    close_match_google_addresses = set(close_matches_df['cleaned_address_google'])
    google_only = google_only[~google_only['cleaned_address'].isin(close_match_google_addresses)]

google_only['data_source'] = 'google_only'
google_only['match_type'] = 'unique'
google_only['similarity_score'] = 0.0
google_only['match_reason'] = 'only_in_google'

admin_only = remaining_groundtruth.copy()
if not close_matches_df.empty:
    close_match_groundtruth_addresses = set(close_matches_df['cleaned_address_groundtruth'])
    admin_only = admin_only[~admin_only['cleaned_address'].isin(close_match_groundtruth_addresses)]

admin_only['data_source'] = 'admin_only'
admin_only['match_type'] = 'unique'
admin_only['similarity_score'] = 0.0
admin_only['match_reason'] = 'only_in_admin'

print(f"Google-only pools: {len(google_only)}")
print(f"Admin-only pools: {len(admin_only)}")

# Step 5: Save results
print("\nSaving results...")

if not exact_matches.empty:
    exact_matches.to_csv('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/public_swimmingpools_germany/exact_matches_strict.csv', index=False)
    print(f"Saved {len(exact_matches)} exact matches to 'exact_matches_strict.csv'")

if not close_matches_df.empty:
    close_matches_df.to_csv('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/public_swimmingpools_germany/close_matches_strict.csv', index=False)
    print(f"Saved {len(close_matches_df)} STRICT close matches to 'close_matches_strict.csv'")

if not google_only.empty:
    google_only.to_file('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/public_swimmingpools_germany/google_only_pools_strict.gpkg', driver='GPKG')
    google_only_csv = google_only.drop(columns=['geometry']) if 'geometry' in google_only.columns else google_only
    google_only_csv.to_csv('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/public_swimmingpools_germany/google_only_pools_strict.csv', index=False)
    print(f"Saved {len(google_only)} Google-only pools")

if not admin_only.empty:
    admin_only.to_csv('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/public_swimmingpools_germany/admin_only_pools_strict.csv', index=False)
    print(f"Saved {len(admin_only)} admin-only pools")

# Step 6: Summary with quality metrics
print("\n=== STRICT MATCHING SUMMARY ===")
print(f"Total Google Maps pools: {len(pools_clean)}")
print(f"Total Ground Truth pools: {len(groundtruth_clean)}")
print(f"Exact matches: {len(exact_matches)}")
print(f"STRICT close matches: {len(close_matches_df)}")
print(f"Google-only pools: {len(google_only)}")
print(f"Admin-only pools: {len(admin_only)}")

# Quality metrics for close matches
if not close_matches_df.empty:
    avg_similarity = close_matches_df['similarity_score'].mean()
    min_similarity = close_matches_df['similarity_score'].min()
    print(f"\nClose match quality:")
    print(f"  Average similarity: {avg_similarity:.3f}")
    print(f"  Minimum similarity: {min_similarity:.3f}")
    
    # Show reasons for close matches
    reason_counts = close_matches_df['match_reason'].value_counts().head(5)
    print(f"  Top match reasons:")
    for reason, count in reason_counts.items():
        print(f"    - {reason}: {count}")

# Show examples of close matches
if not close_matches_df.empty:
    print(f"\n=== SAMPLE STRICT CLOSE MATCHES ===")
    samples = close_matches_df.head(3)
    for idx, row in samples.iterrows():
        print(f"Match {idx+1}:")
        print(f"  Google: {row['cleaned_address_google']}")
        print(f"  Ground: {row['cleaned_address_groundtruth']}")
        print(f"  Similarity: {row['similarity_score']:.3f}")
        print(f"  Reason: {row['match_reason']}")
        print()

In [ ]:
groundtruth_pools['address'] = groundtruth_pools['street'] + ', ' + groundtruth_pools['postal_code'].astype(str) + ' ' + groundtruth_pools['city'] + ', Germany'

In [ ]:
groundtruth_pools[['address']]

In [ ]:
import pandas as pd
import re
from typing import Dict, Any

def clean_german_address(address: str) -> str:
    """
    Clean German addresses by normalizing common variations while preserving structure.
    """
    if pd.isna(address):
        return ""
    
    # Convert to string and lowercase for consistent processing
    address = str(address).lower().strip()
    
    # Remove country suffix if present
    address = re.sub(r',\s*germany\s*$', '', address, flags=re.IGNORECASE)
    
    # Normalize German umlauts and special characters
    replacements = {
        'ä': 'ae', 'ö': 'oe', 'ü': 'ue', 'ß': 'ss',
        'á': 'a', 'é': 'e', 'í': 'i', 'ó': 'o', 'ú': 'u'
    }
    
    for old, new in replacements.items():
        address = address.replace(old, new)
    
    # Standardize common street abbreviations
    street_abbreviations = {
        r'\bstr\.': 'strasse',
        r'\bstr\b': 'strasse',
        r'\bst\.': 'strasse',
        r'\bstrassen\b': 'strasse',
        r'\bstraße\b': 'strasse',
        r'\bplatz\b': 'pl',
        r'\ballee\b': 'al',
        r'\bweg\b': 'weg',
        r'\bavenue\b': 'av',
        r'\bboulevard\b': 'blvd',
    }
    
    for pattern, replacement in street_abbreviations.items():
        address = re.sub(pattern, replacement, address)
    
    # Remove punctuation except spaces, numbers, and letters
    address = re.sub(r'[^\w\s]', '', address)
    
    # Normalize whitespace
    address = re.sub(r'\s+', ' ', address).strip()
    
    return address

def extract_address_components(address: str) -> Dict[str, Any]:
    """
    Extract key components from German addresses for more robust matching.
    """
    components = {
        'full_address': address,
        'street': '',
        'house_number': '',
        'postal_code': '',
        'city': ''
    }
    
    # Try to extract postal code (5 digits)
    postal_match = re.search(r'(\d{5})', address)
    if postal_match:
        components['postal_code'] = postal_match.group(1)
    
    # Common pattern for German addresses: Street HouseNumber, PostalCode City
    pattern1 = r'^([a-z\s]+)\s+(\d+[a-z]?)\s*,\s*(\d{5})\s+([a-z\s]+)$'
    pattern2 = r'^([a-z\s]+)\s+(\d+[a-z]?)\s+(\d{5})\s+([a-z\s]+)$'
    
    for pattern in [pattern1, pattern2]:
        match = re.match(pattern, address)
        if match:
            components['street'] = match.group(1).strip()
            components['house_number'] = match.group(2).strip()
            components['postal_code'] = match.group(3).strip()
            components['city'] = match.group(4).strip()
            break
    
    return components

In [ ]:
def process_and_match_datasets(google_df: pd.DataFrame, admin_df: pd.DataFrame, 
                             google_address_col: str = 'address',
                             admin_address_col: str = 'address') -> Dict[str, pd.DataFrame]:
    """
    Process both datasets and perform the three types of matches.
    
    Returns:
        Dictionary with keys:
        - 'in_both': Rows present in both datasets
        - 'google_only': Rows only in Google dataset
        - 'admin_only': Rows only in Administrative dataset
    """
    
    # Create copies to avoid modifying original data
    google_clean = google_df.copy()
    admin_clean = admin_df.copy()
    
    # Clean addresses in both datasets
    google_clean['cleaned_address'] = google_clean[google_address_col].apply(clean_german_address)
    admin_clean['cleaned_address'] = admin_clean[admin_address_col].apply(clean_german_address)
    
    # Extract components for more robust matching
    google_components = google_clean['cleaned_address'].apply(extract_address_components)
    admin_components = admin_clean['cleaned_address'].apply(extract_address_components)
    
    # Create component columns
    for component in ['street', 'house_number', 'postal_code', 'city']:
        google_clean[f'clean_{component}'] = google_components.apply(lambda x: x[component])
        admin_clean[f'clean_{component}'] = admin_components.apply(lambda x: x[component])
    
    # Create matching keys
    # Key 1: Full cleaned address
    google_clean['match_key_full'] = google_clean['cleaned_address']
    admin_clean['match_key_full'] = admin_clean['cleaned_address']
    
    # Key 2: Street + House number + Postal code (more robust)
    google_clean['match_key_components'] = (
        google_clean['clean_street'] + ' ' + 
        google_clean['clean_house_number'] + ' ' + 
        google_clean['clean_postal_code']
    )
    admin_clean['match_key_components'] = (
        admin_clean['clean_street'] + ' ' + 
        admin_clean['clean_house_number'] + ' ' + 
        admin_clean['clean_postal_code']
    )
    
    # Perform matching using component key (more reliable)
    google_clean['in_admin'] = google_clean['match_key_components'].isin(admin_clean['match_key_components'])
    admin_clean['in_google'] = admin_clean['match_key_components'].isin(google_clean['match_key_components'])
    
    # Create the three result datasets
    in_both_google = google_clean[google_clean['in_admin']].copy()
    in_both_admin = admin_clean[admin_clean['in_google']].copy()
    
    # Merge the matched datasets
    in_both = pd.merge(
        in_both_google, 
        in_both_admin, 
        left_on='match_key_components', 
        right_on='match_key_components',
        suffixes=('_google', '_admin')
    )
    
    google_only = google_clean[~google_clean['in_admin']].copy()
    admin_only = admin_clean[~admin_clean['in_google']].copy()
    
    return {
        'in_both': in_both,
        'google_only': google_only,
        'admin_only': admin_only
    }

def create_final_output(results: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    """
    Create a comprehensive output with all matches and source indicators.
    """
    output_dfs = []
    
    # Add in_both matches
    if not results['in_both'].empty:
        both_df = results['in_both'].copy()
        both_df['match_type'] = 'in_both'
        output_dfs.append(both_df)
    
    # Add google_only
    if not results['google_only'].empty:
        google_only_df = results['google_only'].copy()
        google_only_df['match_type'] = 'google_only'
        output_dfs.append(google_only_df)
    
    # Add admin_only
    if not results['admin_only'].empty:
        admin_only_df = results['admin_only'].copy()
        admin_only_df['match_type'] = 'admin_only'
        output_dfs.append(admin_only_df)
    
    if output_dfs:
        return pd.concat(output_dfs, ignore_index=True)
    else:
        return pd.DataFrame()

# Example usage
def main():
    # Load your datasets
    google_df = pd.read_csv('google_maps_data.csv')
    admin_df = pd.read_csv('administrative_data.csv')
    
    # Process and match
    results = process_and_match_datasets(google_df, admin_df)
    
    # Create final output
    final_output = create_final_output(results)
    
    # Save results
    final_output.to_csv('address_matching_results.csv', index=False)
    
    # Also save individual datasets
    results['in_both'].to_csv('in_both_datasets.csv', index=False)
    results['google_only'].to_csv('google_only_pools.csv', index=False)
    results['admin_only'].to_csv('admin_only_locations.csv', index=False)
    
    # Print summary
    print(f"Total matches found: {len(results['in_both'])}")
    print(f"Google only: {len(results['google_only'])}")
    print(f"Administrative only: {len(results['admin_only'])}")
    
    return results

In [ ]:
# Test the cleaning function with your examples
test_addresses = [
    "Am Hangeweiher 32, 52074 Aachen, Germany",
    "Elisabethstraße 10, 52062 Aachen, Germany",
    "Elisabethstrasse 10, 52062 Aachen, Germany",  # Alternative spelling
    "Am Hangeweiher 32, 52074 Aachen"  # Without country
]

for addr in test_addresses:
    cleaned = clean_german_address(addr)
    components = extract_address_components(cleaned)
    print(f"Original: {addr}")
    print(f"Cleaned: {cleaned}")
    print(f"Components: {components}")
    print("-" * 50)

In [ ]:
import pandas as pd
import re
from rapidfuzz import fuzz, process

# ==========================
# STEP 1 — Normalize addresses
# ==========================
def normalize_address(addr):
    """Normalize casing, umlauts, ß, abbreviations, and spacing in German addresses."""
    if pd.isna(addr):
        return ''
    addr = addr.lower().strip()
    addr = re.sub(r'\s+', ' ', addr)
    
    # Replace German characters and unify abbreviations
    replacements = {
        'ä': 'ae',
        'ö': 'oe',
        'ü': 'ue',
        'ß': 'ss',
        'straße': 'strasse',
        'str.': 'strasse',
    }
    for k, v in replacements.items():
        addr = addr.replace(k, v)
    
    # Remove punctuation and unify spacing
    addr = re.sub(r'[.,;:/\\-]', ' ', addr)
    addr = re.sub(r'\s+', ' ', addr).strip()
    return addr


# ==========================
# STEP 2 — Extract postal code, house number, and street
# ==========================
def extract_postal_code(addr):
    # German postal codes are 5 digits
    m = re.search(r'\b\d{5}\b', addr)
    return m.group(0) if m else None

def extract_number(addr):
    nums = re.findall(r'\d+', addr)
    return nums[0] if nums else None

def extract_street(addr):
    # everything before the first number
    parts = re.split(r'\d+', addr)
    return parts[0].strip() if parts else addr


# ==========================
# STEP 3 — Apply normalization and extraction
# ==========================
for df in [pools, groundtruth_pools]:
    df['norm_address'] = df['address'].apply(normalize_address)
    df['postal_code'] = df['norm_address'].apply(extract_postal_code)
    df['house_num'] = df['norm_address'].apply(extract_number)
    df['street_part'] = df['norm_address'].apply(extract_street)


# ==========================
# STEP 4 — Fuzzy matching with postal code + house number constraints
# ==========================
merged_candidates = []

# Group by postal code + house number to enforce strict constraints
for (plz, num), p_subset in pools.groupby(['postal_code', 'house_num']):
    if pd.isna(plz) or pd.isna(num):
        continue

    g_subset = groundtruth_pools[
        (groundtruth_pools['postal_code'] == plz) &
        (groundtruth_pools['house_num'] == num)
    ]
    if g_subset.empty:
        continue

    # Compare street parts only
    for _, p_row in p_subset.iterrows():
        best_match = process.extractOne(
            p_row['street_part'],
            g_subset['street_part'],
            scorer=fuzz.token_sort_ratio
        )

        # Accept only high-confidence matches
        if best_match and best_match[1] >= 90:
            g_match = g_subset[g_subset['street_part'] == best_match[0]].iloc[0]
            merged_candidates.append({
                **p_row.to_dict(),
                **{f"{k}_grdth": v for k, v in g_match.to_dict().items()},
                'match_score': best_match[1]
            })


# ==========================
# STEP 5 — Combine and inspect
# ==========================
in_both_data = pd.DataFrame(merged_candidates)
in_both_data.info()


In [ ]:
in_both_data.to_csv('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/gmaps_pools_in_baederleben.csv', index=False)

In [ ]:
pools_not_in_groundtruth = pools[~pools['address'].isin(in_both_data['address'])].copy()
pools_not_in_groundtruth.info()
pools_not_in_groundtruth.to_csv('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/gmaps_pools_not_in_baederleben.csv', index=False)

In [ ]:
groundtruth_not_in_pools = groundtruth_pools[~groundtruth_pools['address'].isin(in_both_data['address'])].copy()
groundtruth_not_in_pools.info()
groundtruth_not_in_pools.to_csv('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/baederleben_pools_not_in_gmaps.csv', index=False)

In [ ]:
def selecting_swimming_pois(df: pd.DataFrame) -> pd.DataFrame:
    # Make everything lowercase for comparison
    df["primary_place_type"] = df["primary_place_type"].str.lower()
    df["place_name"] = df["place_name"].str.lower()

    # Define regex patterns
    pattern1 = r"swimming\s+pool$|swimming\s+bath$"
    pattern2 = r"sports\s+complex$|sports\s+centre$|sports\s+hall|aquatic\s+centre$"
    pattern3 = r"\b(swimming pool$|swimming|swimming bath|totobad|hallenbad|freibad|schwimmbad|kombibad)\b"

    # Apply filtering
    mask = (
        df["primary_place_type"].str.contains(pattern1, regex=True, na=False)
        | (
            df["primary_place_type"].str.contains(pattern2, regex=True, na=False)
            & df["place_name"].str.contains(pattern3, regex=True, na=False)
        )
    )

    swimming_pois = df.loc[mask, [
        "place_id", 
        "place_name",
        "address", 
        "borough", 
        "place_name", 
        "primary_place_type", 
        "place_subtype", 
        "total_reviews", 
        "place_rating",
        "google_url"
    ]].copy()


    print(f"Selected {len(swimming_pois)} swimming-related POIs.")
    return swimming_pois

In [ ]:
swimming_pools = selecting_swimming_pois(pools)


In [ ]:
pools_verification = pools[['place_name', 'LAND_NAME', 'WKR_NAME', 'longitude', 'latitude', 'address', 'primary_place_type', 'place_subtype', 'total_reviews', 'place_rating', 'google_url', 'website']].to_csv('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/germany_swimmingpools_verification.csv', index=False)

In [ ]:
kw_swim = r"indoor/?outdoor swimming pool|swimming pool|aquatic center"
kw_exclude = ("hotel", "spa", "fitness center", "european_restaurant", "landscaper")

selection = pools[
    (pools["LAND_NAME"] == "Saarland")
    & (pools["WKR_NAME"] == "Saarbrücken")
    & pools["primary_place_type"].str.contains(kw_swim, case=False, na=False)
    & ~pools["place_subtype"].fillna("").str.lower().str.contains("|".join(kw_exclude))
]

selection

In [ ]:
def selecting_swimming_pois(df: pd.DataFrame) -> pd.DataFrame:
    # Make everything lowercase for comparison
    df["primary_place_type"] = df["primary_place_type"].str.lower()
    df["place_name"] = df["place_name"].str.lower()

    # Define regex patterns
    pattern1 = r"swimming\s+pool$|swimming\s+bath$"
    pattern2 = r"sports\s+complex$|sports\s+centre$|sports\s+hall|aquatic\s+centre$"
    pattern3 = r"\b(swimming pool$|swimming|swimming bath|totobad|hallenbad|freibad|schwimmbad|kombibad)\b"

    # Apply filtering
    mask = (
        df["primary_place_type"].str.contains(pattern1, regex=True, na=False)
        | (
            df["place_subtype"].str.contains(pattern2, regex=True, na=False)
            & df["place_name"].str.contains(pattern3, regex=True, na=False)
        )
    )

    swimming_pois = df.loc[mask, [
        "place_id", 
        "place_name",
        "address", 
        "borough", 
        "place_name", 
        "primary_place_type", 
        "place_subtype", 
        "total_reviews", 
        "place_rating",
        "google_url"
    ]].copy()


    print(f"Selected {len(swimming_pois)} swimming-related POIs.")
    return swimming_pois


In [ ]:
swimming_pools = selecting_swimming_pois(pools)
swimming_pools.info()

In [ ]:
swimming_pools.head()

### reviews

In [ ]:
pool_reviews = pd.read_parquet('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/public_swimmingpools_germany/clean_datasets/merged_reviews.parquet')


In [ ]:
pool_reviews.info()

In [ ]:
pool_reviews =  pool_reviews.drop_duplicates(subset=['review_id']).reset_index(drop=True)
pool_reviews.info()

## reversed 419

In [ ]:
import pandas as pd
df_419 = pd.read_parquet("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/GERMAN_PUBLIC_POOLS/rev_419/general_poi_info.parquet")
df_419.info()

In [ ]:
df_419.to_csv('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/GERMAN_PUBLIC_POOLS/rev_419/pois_419.csv', index=False)